# Session 5: Introduction to MCP

**Learning goals:**

- Explain how MCP reduces repeated tool integration work in AI applications.
- Identify the host, client and server, and distinguish tools, resources and prompts.
- Compare stdio with Streamable HTTP and trace a tool discovery and invocation exchange.
- Build a weather MCP server and connect a Python client to it.

**Prerequisites:**

- Python functions, dictionaries, JSON, HTTP and Function Calling.
- Use a WeatherAPI key for the custom weather tool.
- Have an LLM API key and a model that supports Function Calling.


## Outline

1. From Function Calling to MCP
2. MCP architecture and core primitives
3. Transport in MCP
4. Build your own MCP server
5. Connect MCP to the chatbot
6. Multi-tool MCP chatbot with stdio


## 1. From Function Calling to MCP

In [Sessions 3–4](<Session 3-4 Building a Chatbot with an LLM API.ipynb>), we implemented a weather function, wrote its JSON Schema, and added a function mapping and dispatch code. Each application needed its own tool integration code. With more applications and tools, that work was repeated.

The **Model Context Protocol (MCP)** provides a common interface for discovering and invoking external capabilities. Tool implementations live in reusable servers, which compatible applications access through MCP clients.

- **Discover tools:** `tools/list` returns tool names, descriptions and input schemas. FastMCP can generate these schemas from Python type hints and docstrings.
- **Call tools:** `tools/call` sends a tool name and arguments to the server for execution.
- **Reuse implementations:** multiple applications can connect to the same tool server instead of maintaining separate copies of its service-specific code.

**Function Calling and MCP work together:** the model proposes a tool call; the host checks permissions and routes it through MCP. The host still converts tool descriptions to its LLM provider's format and returns results with the matching call ID.

In the figure, both designs use the same two APIs. Without MCP, two applications maintain four adapter implementations. With MCP, they reuse two adapters held in shared servers. The orange blocks show where the service-specific code lives.

[Official MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture)


![Applications duplicate service-specific adapters without MCP; with MCP, they reuse adapters in shared servers.](assets/session5/before-after-mcp.png)


## 2. MCP architecture

| Role | Responsibility | Connection to earlier sessions |
|---|---|---|
| Host | Owns the user interface, LLM conversation, server selection and permissions | The chatbot application from Sessions 3–4 |
| Client | Handles protocol communication with one server | A protocol-aware connection inside the chatbot |
| Server | Publishes and implements tools, resources and prompts | Similar service responsibility to Session 2's FastAPI backend |

One host can create several clients, each with a one-to-one connection to a server. A server process can also serve other clients when its transport and deployment support that. The LLM does not directly execute Python: the host routes an allowed request through a client.

![An AI host contains two clients, each connected to one server exposing tools, resources and prompts.](assets/session5/mcp-architecture.png)

In the diagram, Client A connects to the weather server over stdio. Client B connects to an illustrative document server over Streamable HTTP, labelled HTTP in the figure. Each server groups its capabilities into tools, resources and prompts. A server may expose any supported subset; it does not have to provide all three. The weather examples match the server we build later; the document server illustrates a second connection. Arrows from the LLM represent requests routed by the host, which controls execution.

The host controls which servers and data enter a conversation. For example, a document server and a weather server need separate permissions even when they share the same protocol. A server can wrap a REST API internally; MCP does not replace that upstream API.

The practical weather example below uses HTTP so the server and client can run in separate terminals.


### 2.1 Tools, resources and prompts

| Primitive | What it offers | Common control pattern | Example |
|---|---|---|---|
| Tool | An executable operation, including reads or writes | Model proposes; host authorises and invokes | `get_weather(city="Perth")` |
| Resource | Addressable context read by URI | Application selects context | `info://server-status` |
| Prompt | A reusable message template with arguments | User selects a workflow through the host | A weather briefing template |

Resources provide data without a write operation; their content can still change over time. A resource URI identifies content and need not be a web URL. Resources may be fixed (`info://server-status`) or templated (`docs://documents/{doc_id}`). Text, JSON and binary data use appropriate MIME types.

A prompt can produce one or more messages. Retrieving it does not call the LLM or run its suggested tools. The host decides how to use those messages.

Tools are our main focus because they connect to the previous chatbot. We will also read one resource and retrieve one prompt to make the distinctions concrete.

[MCP server concepts](https://modelcontextprotocol.io/docs/learn/server-concepts)


![The MCP client invokes tools, reads resources and retrieves prompts exposed by a server; each capability is illustrated with examples.](assets/session5/mcp-client-server-primitives.png)


## 3. Transport in MCP

A **transport** carries messages between an MCP client and server. There are three approaches to recognise:

- **stdio (standard input/output):** used for local servers. The client starts the server as a subprocess, sends messages to its standard input and reads replies from its standard output.
- **HTTP+SSE (Server-Sent Events):** an older approach for remote servers. The client sends messages using HTTP POST, while the server sends messages back through a separate SSE connection.
- **Streamable HTTP:** used for network-accessible servers. The client sends messages to one MCP endpoint using HTTP POST, and the server returns JSON or streams messages using SSE. It replaces the older HTTP+SSE transport.

[Official MCP transport specification](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports)

![Compare local subprocess pipes and an independent HTTP server.](assets/session5/mcp-transport.png)

**Communication lifecycle: two protocol versions**

Both versions use `tools/list` to discover available tools and `tools/call` to invoke one. The main difference is how protocol information is exchanged before and during those requests.

- **MCP 2025-11-25:** the client first sends `initialize`, receives the server's version and capabilities, then sends `notifications/initialized`. It can then list and call tools. Communication ends by closing the transport.
- **MCP 2026-07-28:** there is no required initialisation handshake. The client can optionally call `server/discover`; each request carries its protocol version and client capabilities. HTTP requests complete independently, with no protocol session to terminate. A local stdio connection ends by closing its pipes.

Clients supporting both versions can use the older handshake when connecting to a legacy server. The lower row is a teaching sequence, rather than three mandatory protocol phases.

[Session-based flow (2025-11-25)](https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle) · [Per-request metadata and compatibility (2026-07-28)](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning)

![MCP 2025-11-25 uses an initialisation handshake; MCP 2026-07-28 uses per-request metadata. Both versions list and call tools.](assets/session5/mcp-communication-lifecycle.png)


## 4. Build your own MCP server

Install the required libraries in your Python environment:

```bat
python -m pip install fastmcp requests
```

Use the existing WeatherAPI request in `get_weather(city)` to return `city`, `current_temp` and `condition`. The `@mcp.tool()` decorator exposes the function through MCP and generates its input schema.

The code below defines the server, including a status resource and briefing prompt. Set `WEATHER_API_KEY` in the Python file before calling the weather tool. Running this cell defines the capabilities; start the HTTP server in Terminal 1 using the instructions below.


In [ ]:
"""Expose WeatherAPI through a local MCP server for Session 5."""

import requests
from fastmcp import FastMCP

WEATHER_API_KEY = "YOUR_WEATHERAPI_KEY"  # Replace with your WeatherAPI key.
CURRENT_WEATHER_URL = "https://api.weatherapi.com/v1/current.json"
mcp = FastMCP(name="Course Weather Server")


@mcp.tool()
def get_weather(city: str) -> dict:
    """Return simplified weather for a city using WeatherAPI.

    Args:
        city: A city name, such as Perth or Sydney; include the country if needed.
    """
    city = city.strip()
    if not city:
        raise ValueError("Enter a city name.")
    api_key = WEATHER_API_KEY.strip()
    if not api_key or api_key == "YOUR_WEATHERAPI_KEY":
        raise ValueError("Replace WEATHER_API_KEY in weather_mcp_server.py before calling get_weather.")

    # key authenticates the request; q selects a location; aqi omits air-quality data.
    params = {"key": api_key, "q": city, "aqi": "no"}
    response = requests.get(
        CURRENT_WEATHER_URL, params=params, timeout=10  # Wait up to 10 seconds.
    )
    response.raise_for_status()
    data = response.json()

    # Flatten the location and current objects into one small record.
    return {
        "city": data["location"]["name"],
        "current_temp": data["current"]["temp_c"],
        "condition": data["current"]["condition"]["text"],
    }


@mcp.resource("info://server-status", mime_type="text/plain")
def server_status() -> str:
    """Describe this server without making a weather request."""
    return "Course Weather Server. Data source: WeatherAPI. A key is needed for live weather."


@mcp.prompt()
def weather_briefing(city: str) -> str:
    """Prepare a reusable request for a short weather briefing."""
    return f"Use current weather for {city} to write a short briefing. Include the city, temperature and conditions."


**Run the server in Terminal 1**

On the TAFE Windows computers, run [setup_mcp_ss.bat](practice/session5/setup_mcp_ss.bat) to prepare the Python environment.

[Download weather_mcp_server.py](practice/session5/weather_mcp_server.py) into your working directory. Enter your WeatherAPI key in the file. In Terminal 1, activate the environment and run:

```bat
python weather_mcp_server.py
```

Keep this terminal open. The server listens at `http://127.0.0.1:8000/mcp` using **Streamable HTTP**. A client in Terminal 2 connects to that address. Press **Ctrl+C** in Terminal 1 when you want to stop the server.

[FastMCP HTTP server guide](https://gofastmcp.com/deployment/running-server)


## 5. Connect MCP to the chatbot

Run the weather MCP server in Terminal 1. We will connect to it over HTTP, reuse the earlier chatbot's conversation loop, and run the client in Terminal 2.

The chatbot host uses two separate connections: an LLM API for messages and tool-call decisions, and an MCP client for discovering and calling the weather tool. The server is built with FastMCP; its `get_weather(city)` tool calls the Weather API through a REST request.

The host sends messages and tool schemas to the LLM. When the LLM requests weather data, the host calls the MCP server, receives the tool result, and sends it back to the LLM to produce the final answer.

![The chatbot host connects to an LLM through the OpenRouter API and to a FastMCP weather server through Streamable HTTP; the server calls the Weather API and returns the tool result.](assets/session5/mcp-weather-chatbot-connection.png)

### 5.1 MCP client

The client connects only to the weather server at `http://127.0.0.1:8000/mcp`. Start the server before running the following cells. Passing the URL to `Client` opens an HTTP connection to the existing process.


In [ ]:
import json
from fastmcp import Client

SERVER_URL = "http://127.0.0.1:8000/mcp"

def tool_schema(tool):
    """Read the input schema exposed by the installed MCP SDK."""
    if hasattr(tool, "input_schema"):
        return tool.input_schema
    return tool.inputSchema

async with Client(SERVER_URL) as client:
    weather_tools = await client.list_tools()  # Discover this server's tools.
    for tool in weather_tools:
        print(tool.name, json.dumps(tool_schema(tool), indent=2))


The catalogue includes `get_weather` and its `city` input schema. Enter the WeatherAPI key in the server file and keep the server running in Terminal 1. Restart it if you changed the file. The next cell calls the weather tool directly. The dictionary result is returned as text content for the LLM.


In [ ]:
def result_text(result):
    """Join the text content returned by the weather MCP tool."""
    return "\n".join(block.text for block in result.content if getattr(block, "type", None) == "text")

async with Client(SERVER_URL) as client:
    result = await client.call_tool("get_weather", {"city": "Perth"})  # Request current weather.
    print(result_text(result))


### 5.2 Integrate into chatbot

Reuse the streaming chatbot from [Sessions 3–4](<Session 3-4 Building a Chatbot with an LLM API.ipynb>): its API settings, stream reader, `send_messages()`, message history and `tool_call_id` handling. There are two MCP changes:

- Discover tool declarations from the weather server instead of writing the `tools` schema by hand.
- Call the MCP client in `execute_tool()` instead of using `mapping_tool_function`.

Use the same API endpoint, key and tool-capable model as your earlier chatbot. The example below uses the OpenRouter endpoint; change the settings if you used another compatible provider.


In [ ]:
import requests

CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
CHAT_API_KEY = "YOUR_LLM_API_KEY"  # Use the same LLM API key as your earlier chatbot.
CHAT_MODEL = "YOUR_MODEL_ID"  # Select the same tool-capable model as before.
ALLOWED_TOOLS = {"get_weather"}


Convert the discovered `get_weather` schema into the function declaration format used previously. The host permits that tool and forwards its arguments through MCP.


In [ ]:
async def mcp_tools_to_function_declarations(client):
    """Convert discovered weather tools to the LLM's function declaration format."""
    discovered = await client.list_tools()
    return [
        {"type": "function", "function": {
            "name": tool.name,
            "description": tool.description or tool.name,
            "parameters": tool_schema(tool),
        }}
        for tool in discovered if tool.name in ALLOWED_TOOLS
    ]

async def execute_tool(client, tool_name, tool_args):
    """Call the weather server instead of looking up a local Python function."""
    if tool_name not in ALLOWED_TOOLS:
        raise ValueError("This tool is not enabled by the chatbot.")
    result = await client.call_tool(tool_name, tool_args)
    return result_text(result)


Keep the previous streaming HTTP code. It still prints text as it arrives and assembles tool-call fragments, including provider metadata. Here, `tools` is populated by MCP discovery before the first model request.


In [ ]:
def read_stream_deltas(response):
    """Read text and tool fragments from the API's SSE data lines."""
    # A small chunk_size avoids buffering short replies before displaying them.
    for line in response.iter_lines(chunk_size=1, decode_unicode=True):
        if not line.startswith("data:"):
            continue  # Skip blank lines and keep-alive comments.
        data = line[5:].strip()
        if data == "[DONE]":
            return
        chunk = json.loads(data)
        if chunk.get("error"):
            raise RuntimeError(chunk["error"])
        for choice in chunk.get("choices", []):
            if choice.get("finish_reason") not in (None, "stop", "tool_calls"):
                raise RuntimeError(f"Incomplete model response: {choice['finish_reason']}")
            yield choice.get("delta", {})
    raise RuntimeError("The model response stream ended before [DONE].")


def send_messages(messages):
    """Print streamed text and return the assembled assistant message."""
    # stream requests incremental responses from the model API.
    payload = {"model": CHAT_MODEL, "messages": messages, "tools": tools, "stream": True}
    message = {"role": "assistant", "content": ""}
    tool_calls = {}
    with requests.post(
        CHAT_URL,
        headers={"Authorization": f"Bearer {CHAT_API_KEY}"},
        json=payload,
        stream=True,  # Read the HTTP response incrementally.
        timeout=(10, 60),  # Connection timeout and wait between received data.
    ) as response:
        response.raise_for_status()
        response.encoding = "utf-8"
        for delta in read_stream_deltas(response):
            text = delta.get("content")
            if text:
                if not message["content"]:
                    print("Assistant: ", end="", flush=True)
                print(text, end="", flush=True)
                message["content"] += text

            # Group fragments by index; parallel tool calls may be interleaved.
            for fragment in delta.get("tool_calls") or []:
                index = fragment.get("index", 0)
                if index not in tool_calls:
                    tool_calls[index] = {
                        "id": "",
                        "type": "function",
                        "function": {"name": "", "arguments": ""},
                    }
                call = tool_calls[index]
                for key, value in fragment.items():
                    if key not in ("index", "function"):
                        # Preserve provider metadata, including thought signatures.
                        call[key] = value
                function = fragment.get("function") or {}
                for key in ("name", "arguments"):
                    call["function"][key] += function.get(key) or ""
    if message["content"]:
        print(flush=True)
    if tool_calls:
        message["tool_calls"] = [tool_calls[index] for index in sorted(tool_calls)]
    return message


The conversation loop still sends `pending`, appends the assistant's tool calls, adds the matching tool results and saves the completed turn. It now awaits MCP execution. Blocking HTTP and terminal input run in worker threads so the MCP connection can continue processing messages.


In [ ]:
import asyncio

async def chat(user_message: str, client) -> str:
    """Complete a chat turn, running requested tools through MCP."""
    pending = conversation + [{"role": "user", "content": user_message}]
    while True:
        # Keep the MCP event loop responsive while requests reads the LLM stream.
        response = await asyncio.to_thread(send_messages, pending)
        pending.append(response)  # Preserve the text, tool calls and their IDs.
        if not response.get("tool_calls"):
            break  # A text answer with no tool calls completes the turn.
        for call in response["tool_calls"]:
            tool_name = call["function"]["name"]
            tool_args = json.loads(call["function"]["arguments"])  # Read the JSON arguments.
            print(f"Calling tool {tool_name} with args {tool_args}", flush=True)
            result = await execute_tool(client, tool_name, tool_args)  # Invoke the MCP tool.
            pending.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": result,  # The MCP result has been converted to text.
            })

    conversation[:] = pending  # Save the complete turn after the requests succeed.
    return response["content"]

async def mcp_chat():
    """Keep one weather MCP connection open for the conversation."""
    global tools, conversation
    if not CHAT_API_KEY or CHAT_API_KEY.startswith("YOUR_"):
        raise ValueError("Replace CHAT_API_KEY before starting the chatbot.")
    if not CHAT_MODEL or CHAT_MODEL.startswith("YOUR_"):
        raise ValueError("Replace CHAT_MODEL before starting the chatbot.")

    async with Client(SERVER_URL) as client:
        tools = await mcp_tools_to_function_declarations(client)
        if not tools:
            raise ValueError("The server did not publish the get_weather tool.")
        system_prompt = (
            "You are a weather assistant. Use Australian English. "
            "Do not invent weather data. Ask for a city when it is missing."
        )
        conversation = [{"role": "system", "content": system_prompt}]
        print("Weather chat started. Type quit to finish.")
        while True:
            user_content = (await asyncio.to_thread(input, "You: ")).strip()
            if user_content.lower() == "quit":
                print("Weather chat ended.")
                break
            if not user_content:
                continue
            await chat(user_content, client)  # Print the answer as streamed fragments arrive.


### 5.3 Run in terminal

**HTTP — two terminals**

Use two terminals with the same Python environment activated. Save the downloaded files in your working directory.

**Terminal 1 — server:**

```bat
python weather_mcp_server.py
```

Leave it running. Enter the WeatherAPI key in this file before making live weather requests.

**Terminal 2 — client:** [download mcp_client.py](practice/session5/mcp_client.py) and run:

```bat
python mcp_client.py
```

This lists the weather tool and calls `get_weather` for Perth. To use the chat interface instead, [download mcp_chatbot.py](practice/session5/mcp_chatbot.py), enter `CHAT_API_KEY` and `CHAT_MODEL`, and run it in Terminal 2:

```bat
python mcp_chatbot.py
```

Both clients connect to `http://127.0.0.1:8000/mcp`. Ask the chatbot about Perth, then Sydney. Type `quit` to close the chatbot; the server remains running in Terminal 1.

For the notebook chat below, fill in the settings above and enable the cell while Terminal 1 is running.

**stdio — one terminal**

[Download mcp_client_stdio.py](practice/session5/mcp_client_stdio.py) and save it beside the updated `weather_mcp_server.py`. Enter the WeatherAPI key in the server file, then run:

```bat
python mcp_client_stdio.py
```

This client uses `StdioTransport` to start `python weather_mcp_server.py --transport stdio` as a subprocess. It sends messages through the server's standard input and reads replies from standard output. It discovers the tools, calls `get_weather` for Perth, and closes the subprocess when finished. You do not need to start a server in another terminal or connect to an HTTP URL.

To list tools without making a weather request or entering a WeatherAPI key, run:

```bat
python mcp_client_stdio.py --list-only
```

The HTTP client and chatbot above use the independently running server; the stdio example manages its own server process.


In [ ]:
RUN_CHAT = False  # Set True to enable interactive LLM and weather API requests.

if RUN_CHAT:
    await mcp_chat()  # Use await in a notebook that already has an event loop.
else:
    print("Interactive chat is off. Set RUN_CHAT to True when you are ready.")


## 6. Multi-tool MCP chatbot with stdio

This chatbot combines weather enquiries and research-paper search in one conversation. It discovers four tools from an MCP server and calls them over stdio.

| Tool | Functionality |
|---|---|
| `get_weather(city)` | Current temperature and conditions |
| `get_forecast(city, days=1)` | Daily minimum and maximum temperatures for one to three days |
| `get_history(city, dt)` | Daily recorded temperatures for a date in `YYYY-MM-DD` format |
| `search_papers(query, max_results=5)` | arXiv titles, summaries and source links; one to ten results |

**Download and configure**

- [multi_tool_mcp_server.py](practice/session5/multi_tool_mcp_server.py): enter your WeatherAPI key in `WEATHER_API_KEY`.
- [mcp_multi_tool_chatbot.py](practice/session5/mcp_multi_tool_chatbot.py): enter the LLM `CHAT_API_KEY` and a tool-capable `CHAT_MODEL`, for your chosen LLM provider.

Save both files in the same working directory. The existing `setup_mcp_ss.bat` installs `fastmcp`, `requests` and `arxiv`. No arXiv API key is needed. Forecast and historical access depend on your WeatherAPI plan; choose a supported past date for history requests.

**Run in one terminal**

Activate the Python environment and run `python mcp_multi_tool_chatbot.py`. The chatbot starts the stdio server as a subprocess, discovers the four tools and streams the assistant's replies. You do not need a second server terminal. Type `quit` to end the conversation and close the server process.

To check tool discovery without calling an LLM or external data API, run `python mcp_multi_tool_chatbot.py --list-only`.

**Try these requests**

1. “What is the current weather in Perth?” — `get_weather`.
2. “Give me a three-day forecast for Sydney.” — `get_forecast`.
3. “What was the weather in Perth yesterday?” — `get_history`.
4. “Find two arXiv papers about weather forecasting.” — `search_papers`.
5. “Compare current weather in Perth and Sydney.” — two `get_weather` calls.

Watch the printed tool names and arguments. Follow up with another city, date or topic to see how the chatbot keeps the conversation history and chooses a different tool.


Further activities are in the [Session 5 practice notebook](<practice/session5/Session 5 Practice MCP Server and Client.ipynb>).
